## Хинчагов Руслан 
# 14 Домашнее задание

In [32]:
import os
import sys
import re
import random
import subprocess
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
from dataclasses import dataclass

import torch

from IPython.display import display, Markdown
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

os.environ["TOKENIZERS_PARALLELISM"] = "false"


def ensure_package(package_name: str, import_name: Optional[str] = None) -> None:
    """Пытается импортировать пакет и при необходимости установить его через pip."""
    target = import_name or package_name
    try:
        __import__(target)
    except Exception:
        print(f"Устанавливаем пакет: {package_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])


ensure_package("faiss-cpu", "faiss")
ensure_package("sentence-transformers", "sentence_transformers")


try:
    import faiss  # type: ignore
    FAISS_AVAILABLE = True
except Exception as e:
    FAISS_AVAILABLE = False
    print("FAISS недоступен, будет использован fallback на sklearn NearestNeighbors.")
    print("Причина:", repr(e))


print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("FAISS available:", FAISS_AVAILABLE)

RANDOM_STATE = 42

NumPy: 2.0.2
Pandas: 2.3.3
FAISS available: True


In [33]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_STATE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cpu


In [96]:
documents: List[Dict[str, str]] = [
    {
        "doc_id": "doc_01",
        "title": "Что такое HTTP протокол",
        "text": (
            "HTTP (HyperText Transfer Protocol) – протокол прикладного уровня для передачи гипертекста в World Wide Web. "
            "Работает по модели клиент-сервер: клиент (браузер) отправляет запрос (request), сервер отвечает ответом (response). "
            "Методы: GET (получение), POST (создание), PUT (обновление), DELETE (удаление). Статусы: 200 OK, 404 Not Found, "
            "500 Internal Server Error. Заголовки (headers) передают метаданные: Content-Type, Authorization, Cache-Control."
        ),
    },
    {
        "doc_id": "doc_02", 
        "title": "REST API принципы",
        "text": (
            "REST (Representational State Transfer) – архитектурный стиль для веб-сервисов. Ресурсы идентифицируются URI, "
            "операции через HTTP методы. Stateless – каждый запрос содержит всю информацию. HATEOAS – ответы содержат "
            "ссылки на связанные ресурсы. JSON – основной формат обмена данными. Версионирование: /api/v1/users."
        ),
    },
    {
        "doc_id": "doc_03",
        "title": "Git основы ветвление",
        "text": (
            "Git – распределенная система контроля версий. Основные команды: git init, git clone, git add, git commit. "
            "Ветвление: git branch, git checkout -b feature-branch, git merge. Pull Request – запрос на слияние ветки. "
            "Git Flow: main/develop/feature/release/hotfix. .gitignore исключает ненужные файлы."
        ),
    },
    {
        "doc_id": "doc_04",
        "title": "Responsive дизайн основы",
        "text": (
            "Responsive Web Design адаптирует интерфейс под разные экраны. Media queries: @media (max-width: 768px). "
            "Flexible grid: flexbox, CSS Grid. Viewport meta: <meta name='viewport' content='width=device-width'>. "
            "Mobile-first подход: базовые стили для мобильных, затем улучшения для больших экранов."
        ),
    },
    {
        "doc_id": "doc_05",
        "title": "CSS Flexbox руководство",
        "text": (
            "Flexbox – одномерная модель макета. display: flex на контейнере. Основные свойства: flex-direction (row/column), "
            "justify-content (flex-start, center, space-between), align-items (stretch, center). Flex items: flex-grow, "
            "flex-shrink, flex-basis. Order меняет визуальный порядок элементов."
        ),
    },
    {
        "doc_id": "doc_06",
        "title": "CSS Grid система",
        "text": (
            "CSS Grid – двумерная система макета. display: grid. Свойства: grid-template-columns, grid-template-rows, "
            "grid-template-areas. Размещение: grid-column: 1 / 3, grid-row. Авто размещение: grid-auto-flow. "
            "Подходит для сложных макетов, dashboard'ов."
        ),
    },
    {
        "doc_id": "doc_07",
        "title": "JavaScript замыкания",
        "text": (
            "Замыкание – функция, которая запоминает лексическое окружение, где была создана. function outer() { "
            "const x = 10; return function inner() { return x; }; } const fn = outer(); fn() возвращает 10. "
            "Используется в приватных переменных, колбэках, мемоизации."
        ),
    },
    {
        "doc_id": "doc_08",
        "title": "JavaScript async/await",
        "text": (
            "async/await – синтаксический сахар над Promise. async function всегда возвращает Promise. await "
            "приостанавливает выполнение до resolve. try/catch для обработки ошибок. Promise.all для параллельного "
            "выполнения. Деструктуризация: const { data } = await response.json()."
        ),
    },
    {
        "doc_id": "doc_09",
        "title": "WebSocket соединения",
        "text": (
            "WebSocket – двусторонний протокол для real-time связи. const ws = new WebSocket('ws://example.com'); "
            "События: onopen, onmessage, onclose, onerror. Отличается от HTTP: persistent connection, низкий overhead. "
            "Socket.io – библиотека с fallback'ами."
        ),
    },
    {
        "doc_id": "doc_10",
        "title": "Service Worker PWA",
        "text": (
            "Service Worker – прокси между приложением и сетью. Регистрация: navigator.serviceWorker.register('/sw.js'). "
            "События: install, activate, fetch. Кеширование: caches.open(), cache.addAll(). Offline-first стратегии. "
            "Push notifications через Push API."
        ),
    },
    {
        "doc_id": "doc_11",
        "title": "CORS Cross-Origin политика",
        "text": (
            "CORS (Cross-Origin Resource Sharing) – механизм безопасности браузера. preflight запрос OPTIONS для "
            "нестандартных заголовков/методов. Access-Control-Allow-Origin указывает разрешенные домены. "
            "Credentials: withCredentials=true. Proxy решения для разработки."
        ),
    },
    {
        "doc_id": "doc_12",
        "title": "Web Performance метрики",
        "text": (
            "Core Web Vitals: LCP (2.5s), FID (100ms), CLS (0.1). FCP, TTI, Speed Index. Инструменты: Lighthouse, "
            "WebPageTest, PageSpeed Insights. Оптимизация: lazy loading, compression, CDN, caching."
        ),
    },
    {
        "doc_id": "doc_13",
        "title": "SEO для SPA приложений",
        "text": (
            "Single Page Applications плохо индексируются поисковиками. Prerendering, Server-Side Rendering. "
            "Meta теги: title, description, Open Graph. Sitemap.xml, robots.txt. Structured Data JSON-LD. "
            "React Helmet, Next.js Head."
        ),
    },
    {
        "doc_id": "doc_14",
        "title": "CSS Animations анимации",
        "text": (
            "CSS Transitions: transition: all 0.3s ease. Animations: @keyframes spin { from { transform: rotate(0deg) } "
            "to { transform: rotate(360deg) } }. animation: spin 1s linear infinite. will-change: transform. "
            "GPU acceleration."
        ),
    },
    {
        "doc_id": "doc_15",
        "title": "LocalStorage SessionStorage",
        "text": (
            "localStorage – персистентное хранилище, sessionStorage – сессионное. JSON.stringify/parse для объектов. "
            "StorageEvent при изменениях в других вкладках. Ограничение ~5MB. Не для чувствительных данных."
        ),
    },
    {
        "doc_id": "doc_16",
        "title": "Fetch API современный",
        "text": (
            "fetch(url, { method: 'POST', headers: {'Content-Type': 'application/json'}, body: JSON.stringify(data) }). "
            "then(res => res.json()). AbortController для отмены. credentials: 'include' для cookies."
        ),
    },
    {
        "doc_id": "doc_17",
        "title": "DOM манипуляции querySelector",
        "text": (
            "querySelector, querySelectorAll. classList.add/remove/toggle. dataset для data-*. createElement, "
            "appendChild, insertAdjacentHTML. Event delegation на родителе."
        ),
    },
    {
        "doc_id": "doc_18",
        "title": "Intersection Observer",
        "text": (
            "IntersectionObserver следит за видимостью элементов. new IntersectionObserver(callback, { threshold: 0.1 }). "
            "Lazy loading изображений, infinite scroll, animations on scroll."
        ),
    },
    {
        "doc_id": "doc_19",
        "title": "Drag and Drop API",
        "text": (
            "draggable=true. dragstart, dragover, drop события. dataTransfer.setData/getData. preventDefault() "
            "в dragover. File API для файлов."
        ),
    },
    {
        "doc_id": "doc_20",
        "title": "Canvas 2D графика",
        "text": (
            "const canvas = document.getElementById('canvas'); const ctx = canvas.getContext('2d'); ctx.fillRect(10,10,100,100). "
            "Gradients, shadows, text, paths. requestAnimationFrame для анимаций."
        ),
    },
    {
        "doc_id": "doc_21",
        "title": "Web Workers многопоточность",
        "text": (
            "Web Workers выполняют JS в отдельном потоке. new Worker('worker.js'). postMessage(). onmessage. "
            "SharedArrayBuffer для Shared Memory. Коммуникация через structured clone."
        ),
    },
    {
        "doc_id": "doc_22",
        "title": "Content Security Policy",
        "text": (
            "CSP – защита от XSS. Content-Security-Policy: default-src 'self'; script-src 'self' cdn.example.com. "
            "nonce, hash. report-uri для отчетов."
        ),
    },
    {
        "doc_id": "doc_23",
        "title": "Progressive Enhancement",
        "text": (
            "Progressive Enhancement: базовый HTML+CSS работает без JS. Улучшения добавляются JS. Graceful degradation "
            "наоборот. Accessibility, SEO, скорость загрузки."
        ),
    },
    {
        "doc_id": "doc_24",
        "title": "WebAssembly WASM",
        "text": (
            "WebAssembly – бинарный формат для выполнения кода в браузере. Быстрее JS для вычислений. Rust, C++, "
            "AssemblyScript компиляция. WASI для системных вызовов."
        ),
    },
    {
        "doc_id": "doc_25",
        "title": "Browser DevTools секреты",
        "text": (
            "Performance панель: flame chart, network waterfall. Coverage для неиспользуемого CSS/JS. Memory heap "
            "snapshots. Lighthouse встроенный аудит."
        ),
    },
    {
        "doc_id": "doc_26",
        "title": "CDN Content Delivery Network",
        "text": (
            "CDN – сеть прокси-серверов по всему миру. Статические ресурсы ближе к пользователю. Cloudflare, AWS "
            "CloudFront, Vercel Edge. Cache headers: Cache-Control, ETag."
        ),
    },
    {
        "doc_id": "doc_27",
        "title": "HTTP/2 vs HTTP/3",
        "text": (
            "HTTP/2: multiplexing, header compression, server push. HTTP/3: QUIC+UDP, нет head-of-line blocking. "
            "Браузерная поддержка растет."
        ),
    },
    {
        "doc_id": "doc_28",
        "title": "Web Accessibility a11y",
        "text": (
            "WCAG 2.1 стандарты. Семантическая HTML: <nav>, <main>, <article>. ARIA атрибуты. Keyboard navigation, "
            "screen readers. Contrast ratio 4.5:1."
        ),
    },
    {
        "doc_id": "doc_29",
        "title": "Bundle анализаторы",
        "text": (
            "Webpack Bundle Analyzer, Vite Rollup Plugin. Tree shaking удаляет неиспользуемый код. Code splitting "
            "по маршрутам/компонентам. Чанк карты."
        ),
    },
    {
        "doc_id": "doc_30",
        "title": "Micro Frontends архитектура",
        "text": (
            "Micro Frontends – независимые фронтенд приложения, интегрируемые в единый интерфейс. Module Federation "
            "Webpack 5. iframe, Web Components. Team autonomy, independent deploy."
        ),
    },
]


docs_df = pd.DataFrame(documents)
print("Размер корпуса:", len(docs_df))
display(docs_df[["doc_id", "title", "text"]][:6])

Размер корпуса: 30


,doc_id,title,text
0,doc_01,Что такое HTTP протокол,HTTP (HyperText Transfer Protocol) – протокол ...
1,doc_02,REST API принципы,REST (Representational State Transfer) – архит...
2,doc_03,Git основы ветвление,Git – распределенная система контроля версий. ...
3,doc_04,Responsive дизайн основы,Responsive Web Design адаптирует интерфейс под...
4,doc_05,CSS Flexbox руководство,Flexbox – одномерная модель макета. display: f...
5,doc_06,CSS Grid система,CSS Grid – двумерная система макета. display: ...


В качестве предметной области я взял - фронтенд тематику, здесь разумно строить retrieval / mini-RAG для того чтобы можно было удобно и эффективно находить нужные основы и принципы для помощи в разработке фронетнда. 

In [35]:
class EmbeddingBackend:
    def fit_documents(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError


class SentenceTransformersBackend(EmbeddingBackend):
    def __init__(self, model_name: str, device: str = "cpu") -> None:
        from sentence_transformers import SentenceTransformer  # type: ignore

        self.model_name = model_name
        self.model = SentenceTransformer(model_name, device=device)
        self.backend_name = f"SentenceTransformer: {model_name}"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=12,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        return vectors.astype("float32")

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=12,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        return vectors.astype("float32")


class TfidfFallbackBackend(EmbeddingBackend):
    def __init__(self) -> None:
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2), lowercase=True)
        self.backend_name = "TF-IDF fallback"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        vectors = self.vectorizer.fit_transform(texts).toarray()
        return vectors.astype("float32")

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        vectors = self.vectorizer.transform(texts).toarray()
        return vectors.astype("float32")


def build_embedding_backend(
    model_name: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device: str = "cpu",
) -> EmbeddingBackend:
    try:
        backend = SentenceTransformersBackend(model_name=model_name, device=device)
        print("Используем полноценные dense embeddings.")
        print("Бэкэнд:", backend.backend_name)
        return backend
    except Exception as e:
        print("Не удалось загрузить sentence-transformers encoder.")
        print("Причина:", repr(e))
        print("Переключаемся на TF-IDF fallback. Ноутбук останется рабочим,")
        print("но это уже не полноценные dense embeddings.")
        return TfidfFallbackBackend()


embedder = build_embedding_backend(device=device)

Используем полноценные dense embeddings.
Бэкэнд: SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [36]:
def chunk_text(text: str, chunk_size: int = 22, overlap: int = 5) -> List[str]:
    words = text.replace("\n", " ").split()

    if chunk_size <= 0:
        raise ValueError("chunk_size должен быть положительным.")
    if overlap >= chunk_size:
        raise ValueError("overlap должен быть меньше chunk_size.")

    chunks = []
    step = chunk_size - overlap

    for start in range(0, len(words), step):
        chunk_words = words[start : start + chunk_size]
        if not chunk_words:
            continue

        chunks.append(" ".join(chunk_words))

        if start + chunk_size >= len(words):
            break

    return chunks

def build_chunks_dataframe(
    docs: List[Dict[str, str]],
    chunk_size: int = 25,
    overlap: int = 10,
) -> pd.DataFrame:
    rows = []

    for doc in docs:
        chunks = chunk_text(doc["text"], chunk_size=chunk_size, overlap=overlap)
        for chunk_id, chunk in enumerate(chunks):
            rows.append(
                {
                    "doc_id": doc["doc_id"],
                    "title": doc["title"],
                    "chunk_id": chunk_id,
                    "chunk_text": chunk,
                    "n_words": len(chunk.split()),
                }
            )

    return pd.DataFrame(rows)


chunks_df = build_chunks_dataframe(documents, chunk_size=25, overlap=10)

print("Количество чанков:", len(chunks_df))
display(chunks_df.head(10))




Количество чанков: 41


,doc_id,title,chunk_id,chunk_text,n_words
0,doc_01,Что такое HTTP протокол,0,HTTP (HyperText Transfer Protocol) – протокол ...,25
1,doc_01,Что такое HTTP протокол,1,Работает по модели клиент-сервер: клиент (брау...,25
2,doc_01,Что такое HTTP протокол,2,"(получение), POST (создание), PUT (обновление)...",24
3,doc_02,REST API принципы,0,REST (Representational State Transfer) – архит...,25
4,doc_02,REST API принципы,1,методы. Stateless – каждый запрос содержит всю...,24
5,doc_03,Git основы ветвление,0,Git – распределенная система контроля версий. ...,25
6,doc_03,Git основы ветвление,1,"commit. Ветвление: git branch, git checkout -b...",24
7,doc_04,Responsive дизайн основы,0,Responsive Web Design адаптирует интерфейс под...,25
8,doc_04,Responsive дизайн основы,1,"flexbox, CSS Grid. Viewport meta: <meta name='...",19
9,doc_05,CSS Flexbox руководство,0,Flexbox – одномерная модель макета. display: f...,25


chunk_size - создает много мелких чанков, будет хорош для тестов retrieval на коротких фразах 

overlap - сохраняет связь между чанками

In [37]:
chunk_texts = chunks_df["chunk_text"].tolist()
chunk_embeddings = embedder.fit_documents(chunk_texts)

In [38]:
class VectorSearchIndex:
    def __init__(self, dim: int) -> None:
        self.dim = dim
        self.backend_name = None
        self._faiss_index = None
        self._nn_index = None

        if FAISS_AVAILABLE:
            self._faiss_index = faiss.IndexFlatIP(dim)  # type: ignore[name-defined]
            self.backend_name = "FAISS IndexFlatIP"
        else:
            self._nn_index = NearestNeighbors(metric="cosine")
            self.backend_name = "sklearn NearestNeighbors fallback"

    def add(self, vectors: np.ndarray) -> None:
        vectors = vectors.astype("float32")

        if self._faiss_index is not None:
            self._faiss_index.add(vectors)
        else:
            self._nn_index.fit(vectors)

    def search(self, query_vectors: np.ndarray, top_k: int = 5) -> Tuple[np.ndarray, np.ndarray]:
        query_vectors = query_vectors.astype("float32")

        if self._faiss_index is not None:
            scores, indices = self._faiss_index.search(query_vectors, top_k)
            return scores, indices

        distances, indices = self._nn_index.kneighbors(query_vectors, n_neighbors=top_k)
        scores = 1.0 - distances
        return scores, indices


search_index = VectorSearchIndex(dim=chunk_embeddings.shape[1])
search_index.add(chunk_embeddings)

print("Индекс построен.")
print("Бэкэнд индекса:", search_index.backend_name)

Индекс построен.
Бэкэнд индекса: FAISS IndexFlatIP


In [39]:
def search_similar_chunks(query: str, top_k: int = 5) -> pd.DataFrame:
    query_vectors = embedder.encode_queries([query])
    scores, indices = search_index.search(query_vectors, top_k=top_k)

    rows = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        chunk_row = chunks_df.iloc[int(idx)]
        rows.append(
            {
                "rank": rank,
                "doc_id": chunk_row["doc_id"],
                "title": chunk_row["title"],
                "chunk_id": int(chunk_row["chunk_id"]),
                "score": round(float(score), 4),
                "chunk_text": chunk_row["chunk_text"],
            }
        )

    return pd.DataFrame(rows)


faiss_querys = ["Для чего нужен localstorage?", "Что такое react?", "Как работает WebSocket?", "Как написать html страницу с помощью emmet?", "Что такое адаптивная верстка?"]

for i in faiss_querys:
    faiss_results_df = search_similar_chunks(i, top_k=5)
    display(Markdown(f"**Запрос:** {i}"))
    display(faiss_results_df)



**Запрос:** Для чего нужен localstorage?

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,doc_15,LocalStorage SessionStorage,0,0.5876,"localStorage – персистентное хранилище, sessio..."
1,2,doc_02,REST API принципы,0,0.3638,REST (Representational State Transfer) – архит...
2,3,doc_11,CORS Cross-Origin политика,0,0.3396,CORS (Cross-Origin Resource Sharing) – механиз...
3,4,doc_06,CSS Grid система,1,0.2978,"/ 3, grid-row. Авто размещение: grid-auto-flow..."
4,5,doc_09,WebSocket соединения,1,0.2868,"onclose, onerror. Отличается от HTTP: persiste..."


**Запрос:** Что такое react?

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,doc_01,Что такое HTTP протокол,1,0.3222,Работает по модели клиент-сервер: клиент (брау...
1,2,doc_08,JavaScript async/await,0,0.2891,async/await – синтаксический сахар над Promise...
2,3,doc_07,JavaScript замыкания,0,0.2818,"Замыкание – функция, которая запоминает лексич..."
3,4,doc_08,JavaScript async/await,1,0.2399,resolve. try/catch для обработки ошибок. Promi...
4,5,doc_07,JavaScript замыкания,1,0.2067,= 10; return function inner() { return x; }; }...


**Запрос:** Как работает WebSocket?

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,doc_09,WebSocket соединения,0,0.6904,WebSocket – двусторонний протокол для real-tim...
1,2,doc_24,WebAssembly WASM,0,0.5883,WebAssembly – бинарный формат для выполнения к...
2,3,doc_09,WebSocket соединения,1,0.5666,"onclose, onerror. Отличается от HTTP: persiste..."
3,4,doc_01,Что такое HTTP протокол,0,0.5353,HTTP (HyperText Transfer Protocol) – протокол ...
4,5,doc_12,Web Performance метрики,0,0.5020,"Core Web Vitals: LCP (2.5s), FID (100ms), CLS ..."


**Запрос:** Как написать html страницу с помощью emmet?

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,doc_13,SEO для SPA приложений,0,0.4076,Single Page Applications плохо индексируются п...
1,2,doc_16,Fetch API современный,0,0.3409,"fetch(url, { method: 'POST', headers: {'Conten..."
2,3,doc_01,Что такое HTTP протокол,2,0.3338,"(получение), POST (создание), PUT (обновление)..."
3,4,doc_01,Что такое HTTP протокол,0,0.3130,HTTP (HyperText Transfer Protocol) – протокол ...
4,5,doc_24,WebAssembly WASM,0,0.3127,WebAssembly – бинарный формат для выполнения к...


**Запрос:** Что такое адаптивная верстка?

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,doc_05,CSS Flexbox руководство,0,0.4310,Flexbox – одномерная модель макета. display: f...
1,2,doc_06,CSS Grid система,1,0.3928,"/ 3, grid-row. Авто размещение: grid-auto-flow..."
2,3,doc_30,Micro Frontends архитектура,0,0.3809,Micro Frontends – независимые фронтенд приложе...
3,4,doc_18,Intersection Observer,0,0.3586,IntersectionObserver следит за видимостью элем...
4,5,doc_14,CSS Animations анимации,1,0.3351,to { transform: rotate(360deg) } }. animation:...


In [40]:
# сделаю набор контрольных запросов: 

benchmark_queries: List[Dict[str, object]] = [
    {
        "query_id": "q01", 
        "query": "Как работать с git?",
        "relevant_doc_ids": ["doc_03"]
    },
    {
        "query_id": "q02", 
        "query": "Как сделать анимацию в css?",
        "relevant_doc_ids": ["doc_14"]
    },
    {
        "query_id": "q03", 
        "query": "Как в JavaScript работает ассинхронность?",
        "relevant_doc_ids": ["doc_08"]
    },
    {
        "query_id": "q04", 
        "query": "Что такое веб работники?",
        "relevant_doc_ids": ["doc_21"]
    },
    {
        "query_id": "q05", 
        "query": "В чем отличие между http2 и http3?",
        "relevant_doc_ids": ["doc_27"]
    },
    {
        "query_id": "q06", 
        "query": "Для чего нужен DOM манипуляции?",
        "relevant_doc_ids": ["doc_17"]
    },
    {
        "query_id": "q07", 
        "query": "Как работает сео аналитика для спа?",
        "relevant_doc_ids": ["doc_13"]
    },
    {
        "query_id": "q08", 
        "query": "Как работает css грид система?",
        "relevant_doc_ids": ["doc_06"]
    },
    {
        "query_id": "q09", 
        "query": "Че такое cdn?",
        "relevant_doc_ids": ["doc_26"]
    },
    {
        "query_id": "q10", 
        "query": "Чем является webAsembly",
        "relevant_doc_ids": ["doc_24"]
    },
]

benchmark_df = pd.DataFrame(benchmark_queries)
display(benchmark_df)

,query_id,query,relevant_doc_ids
0,q01,Как работать с git?,[doc_03]
1,q02,Как сделать анимацию в css?,[doc_14]
2,q03,Как в JavaScript работает ассинхронность?,[doc_08]
3,q04,Что такое веб работники?,[doc_21]
4,q05,В чем отличие между http2 и http3?,[doc_27]
5,q06,Для чего нужен DOM манипуляции?,[doc_17]
6,q07,Как работает сео аналитика для спа?,[doc_13]
7,q08,Как работает css грид система?,[doc_06]
8,q09,Че такое cdn?,[doc_26]
9,q10,Чем является webAsembly,[doc_24]


In [ ]:
class SentenceTransformersBackend(EmbeddingBackend):
    def __init__(self, model_name: str, device: str = "cpu") -> None:
        from sentence_transformers import SentenceTransformer  # type: ignore

        self.model_name = model_name
        self.model = SentenceTransformer(model_name, device=device)
        self.backend_name = f"SentenceTransformer: {model_name}"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=16,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        return vectors.astype("float32")

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=16,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        return vectors.astype("float32")


class TfidfBackend(EmbeddingBackend):
    def __init__(self) -> None:
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2))
        self.backend_name = "TF-IDF fallback"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        vectors = self.vectorizer.fit_transform(texts).toarray().astype("float32")
        norms = np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-12
        return vectors / norms

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        vectors = self.vectorizer.transform(texts).toarray().astype("float32")
        norms = np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-12
        return vectors / norms


def select_backend(device: str = "cpu") -> EmbeddingBackend:
    try:
        backend = SentenceTransformersBackend(
            model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
            device=device,
        )
        print("Используем dense-модель эмбеддингов.")
        return backend
    except Exception as e:
        print("Dense-модель недоступна, переключаемся на fallback.")
        print("Причина:", repr(e))
        return TfidfBackend()


def build_chunks(
    docs: List[Dict[str, str]],
    chunk_size: int,
    overlap: int,
) -> List[Dict[str, object]]:
    rows: List[Dict[str, object]] = []
    for doc in docs:
        parts = chunk_text(doc["text"], chunk_size=chunk_size, overlap=overlap)
        for chunk_idx, chunk_text_value in enumerate(parts):
            rows.append(
                {
                    "doc_id": doc["doc_id"],
                    "title": doc["title"],
                    "chunk_id": f"{doc['doc_id']}_chunk_{chunk_idx}",
                    "chunk_idx": chunk_idx,
                    "chunk_text": chunk_text_value,
                }
            )
    return rows


@dataclass
class RetrieverArtifacts:
    backend_name: str
    backend: EmbeddingBackend
    chunks_df: pd.DataFrame
    chunk_vectors: np.ndarray
    index: object


def build_retriever(
    docs: List[Dict[str, str]],
    chunk_size: int = 40,
    overlap: int = 10,
    device: str = "cpu",
) -> RetrieverArtifacts:
    chunks = build_chunks(docs, chunk_size=chunk_size, overlap=overlap)
    chunks_df = pd.DataFrame(chunks)

    backend = select_backend(device=device)
    chunk_vectors = backend.fit_documents(chunks_df["chunk_text"].tolist())

    if not FAISS_AVAILABLE:
        raise RuntimeError("FAISS недоступен. Для этого ноутбука ожидается установленный faiss-cpu.")

    index = faiss.IndexFlatIP(chunk_vectors.shape[1])
    index.add(chunk_vectors)

    return RetrieverArtifacts(
        backend_name=backend.backend_name,
        backend=backend,
        chunks_df=chunks_df,
        chunk_vectors=chunk_vectors,
        index=index,
    )


def search_chunks(
    query: str,
    artifacts: RetrieverArtifacts,
    top_k: int = 3,
) -> pd.DataFrame:
    query_vector = artifacts.backend.encode_queries([query]).astype("float32")
    scores, indices = artifacts.index.search(query_vector, top_k)

    rows = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        chunk_row = artifacts.chunks_df.iloc[int(idx)]
        rows.append(
            {
                "rank": rank,
                "score": float(score),
                "doc_id": chunk_row["doc_id"],
                "title": chunk_row["title"],
                "chunk_id": chunk_row["chunk_id"],
                "chunk_text": chunk_row["chunk_text"],
            }
        )
    return pd.DataFrame(rows)


def unique_doc_order(result_df: pd.DataFrame) -> List[str]:
    seen = set()
    ordered = []
    for doc_id in result_df["doc_id"].tolist():
        if doc_id not in seen:
            seen.add(doc_id)
            ordered.append(doc_id)
    return ordered


def evaluate_query(
    query: str,
    relevant_doc_ids: List[str],
    artifacts: RetrieverArtifacts,
    top_k: int = 3,
) -> Dict[str, object]:
    result_df = search_chunks(query, artifacts=artifacts, top_k=top_k)
    predicted_doc_ids = unique_doc_order(result_df)

    hit = int(any(doc_id in predicted_doc_ids for doc_id in relevant_doc_ids))
    recall = sum(doc_id in predicted_doc_ids for doc_id in relevant_doc_ids) / len(relevant_doc_ids)

    first_relevant_rank = None
    for idx, doc_id in enumerate(predicted_doc_ids, start=1):
        if doc_id in relevant_doc_ids:
            first_relevant_rank = idx
            break

    mrr = 0.0 if first_relevant_rank is None else 1.0 / first_relevant_rank

    return {
        "predicted_doc_ids": predicted_doc_ids,
        "hit": hit,
        "recall": recall,
        "first_relevant_rank": first_relevant_rank,
        "mrr": mrr,
        "result_df": result_df,
    }


def evaluate_benchmark(
    benchmark_rows: List[Dict[str, object]],
    artifacts: RetrieverArtifacts,
    top_k: int = 3,
) -> pd.DataFrame:
    rows = []
    for row in benchmark_rows:
        metrics = evaluate_query(
            query=row["query"],
            relevant_doc_ids=row["relevant_doc_ids"],
            artifacts=artifacts,
            top_k=top_k,
        )
        rows.append(
            {
                "query_id": row["query_id"],
                "query": row["query"],
                "relevant_doc_ids": ", ".join(row["relevant_doc_ids"]),
                "predicted_doc_ids": ", ".join(metrics["predicted_doc_ids"]),
                f"hit@{top_k}": metrics["hit"],
                f"recall@{top_k}": metrics["recall"],
                f"MRR@{top_k}": metrics["mrr"],
                "first_relevant_rank": metrics["first_relevant_rank"],
            }
        )
    return pd.DataFrame(rows)

In [42]:
def split_into_sentences(text: str) -> List[str]:
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    return [p.strip() for p in parts if p.strip()]


def pick_best_sentences(query: str, text: str, top_n: int = 2) -> List[str]:
    sentences = split_into_sentences(text)
    if not sentences:
        return []

    vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    matrix = vectorizer.fit_transform([query] + sentences).toarray().astype(np.float32)

    query_vec = matrix[0]
    sentence_vecs = matrix[1:]

    query_norm = np.linalg.norm(query_vec) + 1e-12
    sent_norms = np.linalg.norm(sentence_vecs, axis=1) + 1e-12
    scores = (sentence_vecs @ query_vec) / (sent_norms * query_norm)

    best_idx = np.argsort(-scores)[:top_n]
    return [sentences[i] for i in best_idx if scores[i] > 0]


def answer_without_retrieval(query: str, documents: List[Dict[str, str]]) -> Dict[str, object]:
    doc_texts = [doc["title"] + ". " + doc["text"] for doc in documents]
    vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    matrix = vectorizer.fit_transform(doc_texts + [query]).toarray().astype(np.float32)

    doc_vecs = matrix[:-1]
    query_vec = matrix[-1]

    doc_norms = np.linalg.norm(doc_vecs, axis=1) + 1e-12
    query_norm = np.linalg.norm(query_vec) + 1e-12
    scores = (doc_vecs @ query_vec) / (doc_norms * query_norm)

    best_idx = int(np.argmax(scores))
    best_doc = documents[best_idx]
    best_sentences = pick_best_sentences(query, best_doc["text"], top_n=2)

    if best_sentences:
        answer = " ".join(best_sentences)
    else:
        answer = (
            "Не удалось уверенно извлечь ответ без retrieval по чанкам. "
            "Система выбрала наиболее похожий документ целиком."
        )

    return {
        "answer": answer,
        "selected_doc_id": best_doc["doc_id"],
        "selected_title": best_doc["title"],
        "score": float(scores[best_idx]),
    }


In [97]:
retrieval_results = []
baseline_chunk_size_1 = 25
baseline_overlap_1 = 10

retrieved_before = []

baseline_chunk_size_2 = 10
baseline_overlap_2 = 3

artifacts_1 = build_retriever(
    documents,
    chunk_size=baseline_chunk_size_1,
    overlap=baseline_overlap_1,
    device=device,
)

artifacts_2 = build_retriever(
    documents,
    chunk_size=baseline_chunk_size_2,
    overlap=baseline_overlap_2,
    device=device,
)


print(f"Для эксперимента 1:\nbaseline_chunk_size: {baseline_overlap_1} | baseline_overlap: {baseline_overlap_1} | top_k: 3")
baseline_eval_k3 = evaluate_benchmark(benchmark_queries, artifacts=artifacts_1, top_k=3)
display(baseline_eval_k3)

summary_k3 = pd.DataFrame(
    {
        "metric": ["mean_hit@3", "mean_recall@3", "mean_MRR@3"],
        "value": [
            baseline_eval_k3["hit@3"].mean(),
            baseline_eval_k3["recall@3"].mean(),
            baseline_eval_k3["MRR@3"].mean(),
        ],
    }
)
display(summary_k3)


print(f"Для эксперимента 2:\nbaseline_chunk_size: {baseline_chunk_size_2} | baseline_overlap: {baseline_overlap_2} | top_k: 5")
baseline_eval_k5 = evaluate_benchmark(benchmark_queries, artifacts=artifacts_2, top_k=5)
display(baseline_eval_k5)

summary_k5 = pd.DataFrame(
    {
        "metric": ["mean_hit@5", "mean_recall@5", "mean_MRR@5"],
        "value": [
            baseline_eval_k5["hit@5"].mean(),
            baseline_eval_k5["recall@5"].mean(),
            baseline_eval_k5["MRR@5"].mean(),
        ],
    }
)
    
display(summary_k5)

for idx, row in baseline_eval_k3.iterrows():
    query = benchmark_queries[idx]
    
    if idx == 5 or idx == 6: 
        retrieved_before.append({
            'query': row.get('query'),
            'before_retrieved_sources': row.get('predicted_doc_ids')
        })
        
    retrieval_results.append({
        'query': row.get('query'),
        'expected_source': row.get('relevant_doc_ids', 'N/A'), 
        'chunk_size': baseline_chunk_size_1,
        'overlap': baseline_overlap_1,
        'retrieved_sources': row.get('predicted_doc_ids', row.get('sources', 'N/A')),
        'hit_at_k': row.get('hit@3'), 
        'mean_recall': row.get('recall@3'),
        'mrr': row.get('MRR@3'), 
        'rank_of_first_relevant': row.get("first_relevant_rank"),
    })

for idx, row in baseline_eval_k5.iterrows():
    query = benchmark_queries[idx]
    
    retrieval_results.append({
        'query': row.get('query'),
        'expected_source': row.get('relevant_doc_ids', 'N/A'), 
        'chunk_size': baseline_chunk_size_2,
        'overlap': baseline_overlap_2,
        'retrieved_sources': row.get('predicted_doc_ids', row.get('sources', 'N/A')),
        'hit_at_k': row.get('hit@5'), 
        'mean_recall': row.get('recall@5'),
        'mrr': row.get('MRR@5'), 
        'rank_of_first_relevant': row.get("first_relevant_rank"),
    })
    
retrieval_eval_df = pd.DataFrame(retrieval_results, columns=["query", "expected_source", "chunk_size", "overlap", "retrieved_sources", "hit_at_k", "mean_recall","mrr","rank_of_first_relevant"])
retrieval_eval_df.to_csv('artifacts/retrieval_eval.csv', index=False, encoding='utf-8')


print(retrieval_eval_df)
print(retrieved_before)

Используем dense-модель эмбеддингов.
Используем dense-модель эмбеддингов.
Для эксперимента 1:
baseline_chunk_size: 10 | baseline_overlap: 10 | top_k: 3


,query_id,query,relevant_doc_ids,predicted_doc_ids,hit@3,recall@3,MRR@3,first_relevant_rank
0,q01,Как работать с git?,doc_03,"doc_03, doc_12",1,1.0,1.0,1.0
1,q02,Как сделать анимацию в css?,doc_14,"doc_14, doc_06, doc_04",1,1.0,1.0,1.0
2,q03,Как в JavaScript работает ассинхронность?,doc_08,"doc_08, doc_21, doc_23",1,1.0,1.0,1.0
3,q04,Что такое веб работники?,doc_21,"doc_21, doc_10, doc_24",1,1.0,1.0,1.0
4,q05,В чем отличие между http2 и http3?,doc_27,"doc_09, doc_27, doc_01",1,1.0,0.5,2.0
5,q06,Для чего нужен DOM манипуляции?,doc_17,"doc_23, doc_13, doc_01",0,0.0,0.0,NaN
6,q07,Как работает сео аналитика для спа?,doc_13,"doc_06, doc_01, doc_25",0,0.0,0.0,NaN
7,q08,Как работает css грид система?,doc_06,"doc_06, doc_23, doc_14",1,1.0,1.0,1.0
8,q09,Че такое cdn?,doc_26,"doc_26, doc_22, doc_12",1,1.0,1.0,1.0
9,q10,Чем является webAsembly,doc_24,"doc_24, doc_09, doc_01",1,1.0,1.0,1.0


,metric,value
0,mean_hit@3,0.80
1,mean_recall@3,0.80
2,mean_MRR@3,0.75


Для эксперимента 2:
baseline_chunk_size: 10 | baseline_overlap: 3 | top_k: 5


,query_id,query,relevant_doc_ids,predicted_doc_ids,hit@5,recall@5,MRR@5,first_relevant_rank
0,q01,Как работать с git?,doc_03,doc_03,1,1.0,1.0,1.0
1,q02,Как сделать анимацию в css?,doc_14,"doc_14, doc_18, doc_06, doc_20",1,1.0,1.0,1.0
2,q03,Как в JavaScript работает ассинхронность?,doc_08,"doc_08, doc_23, doc_15, doc_21, doc_02",1,1.0,1.0,1.0
3,q04,Что такое веб работники?,doc_21,"doc_21, doc_10, doc_24, doc_02, doc_01",1,1.0,1.0,1.0
4,q05,В чем отличие между http2 и http3?,doc_27,"doc_09, doc_27, doc_01, doc_02",1,1.0,0.5,2.0
5,q06,Для чего нужен DOM манипуляции?,doc_17,"doc_23, doc_01, doc_13, doc_14, doc_04",0,0.0,0.0,NaN
6,q07,Как работает сео аналитика для спа?,doc_13,"doc_25, doc_02, doc_06, doc_07",0,0.0,0.0,NaN
7,q08,Как работает css грид система?,doc_06,"doc_06, doc_23, doc_14, doc_01, doc_22",1,1.0,1.0,1.0
8,q09,Че такое cdn?,doc_26,"doc_26, doc_22, doc_02, doc_01, doc_12",1,1.0,1.0,1.0
9,q10,Чем является webAsembly,doc_24,"doc_24, doc_01, doc_09",1,1.0,1.0,1.0


,metric,value
0,mean_hit@5,0.80
1,mean_recall@5,0.80
2,mean_MRR@5,0.75


                                        query expected_source  chunk_size  \
0                         Как работать с git?          doc_03          25   
1                 Как сделать анимацию в css?          doc_14          25   
2   Как в JavaScript работает ассинхронность?          doc_08          25   
3                    Что такое веб работники?          doc_21          25   
4          В чем отличие между http2 и http3?          doc_27          25   
5             Для чего нужен DOM манипуляции?          doc_17          25   
6         Как работает сео аналитика для спа?          doc_13          25   
7              Как работает css грид система?          doc_06          25   
8                               Че такое cdn?          doc_26          25   
9                     Чем является webAsembly          doc_24          25   
10                        Как работать с git?          doc_03          10   
11                Как сделать анимацию в css?          doc_14          10   

In [83]:
# изменю содержание двух документов 

documents[16]["text"] = "DOM-манипуляции (Document Object Model) — это процесс изменения структуры, стиля или контента веб-страницы в реальном времени с помощью JavaScript"

documents[12]["text"] = "SEO (сео) веб-приложений (особенно SPA (спа) — Single Page Applications) работает через обеспечение индексации динамического контента, который создается JavaScript"
documents[12]["title"] = "Сео(seo) аналитика для спа(spa) приложений"

display(docs_df)

,doc_id,title,text
0,doc_01,Что такое HTTP протокол,HTTP (HyperText Transfer Protocol) – протокол ...
1,doc_02,REST API принципы,REST (Representational State Transfer) – архит...
2,doc_03,Git основы ветвление,Git – распределенная система контроля версий. ...
3,doc_04,Responsive дизайн основы,Responsive Web Design адаптирует интерфейс под...
4,doc_05,CSS Flexbox руководство,Flexbox – одномерная модель макета. display: f...
5,doc_06,CSS Grid система,CSS Grid – двумерная система макета. display: ...
6,doc_07,JavaScript замыкания,"Замыкание – функция, которая запоминает лексич..."
7,doc_08,JavaScript async/await,async/await – синтаксический сахар над Promise...
8,doc_09,WebSocket соединения,WebSocket – двусторонний протокол для real-tim...
9,doc_10,Service Worker PWA,Service Worker – прокси между приложением и се...


In [87]:
retrieved_after = []

artifacts_3 = build_retriever(
    documents,
    chunk_size=baseline_chunk_size_1,
    overlap=baseline_overlap_1,
    device=device,
)

baseline_eval_k3 = evaluate_benchmark(benchmark_queries, artifacts=artifacts_3, top_k=3)
display(baseline_eval_k3)

summary_k3 = pd.DataFrame(
    {
        "metric": ["mean_hit@3", "mean_recall@3", "mean_MRR@3"],
        "value": [
            baseline_eval_k3["hit@3"].mean(),
            baseline_eval_k3["recall@3"].mean(),
            baseline_eval_k3["MRR@3"].mean(),
        ],
    }
)
display(summary_k3)
retrieved_after.append({
    "query": "Как работает сео аналитика для спа?",
    "after_retrieved_sources": "doc_06, doc_01, doc_25",
})
retrieved_after.append({
    "query": "Для чего нужен DOM манипуляции?",
    "after_retrieved_sources": "doc_17, doc_13, doc_23",
})

print(retrieved_after)
print(documents[12], documents[16])

Используем dense-модель эмбеддингов.


,query_id,query,relevant_doc_ids,predicted_doc_ids,hit@3,recall@3,MRR@3,first_relevant_rank
0,q01,Как работать с git?,doc_03,"doc_03, doc_12",1,1.0,1.0,1.0
1,q02,Как сделать анимацию в css?,doc_14,"doc_14, doc_06, doc_04",1,1.0,1.0,1.0
2,q03,Как в JavaScript работает ассинхронность?,doc_08,"doc_08, doc_17, doc_13",1,1.0,1.0,1.0
3,q04,Что такое веб работники?,doc_21,"doc_21, doc_10, doc_24",1,1.0,1.0,1.0
4,q05,В чем отличие между http2 и http3?,doc_27,"doc_09, doc_27, doc_01",1,1.0,0.5,2.0
5,q06,Для чего нужен DOM манипуляции?,doc_17,"doc_17, doc_13, doc_23",1,1.0,1.0,1.0
6,q07,Как работает сео аналитика для спа?,doc_13,"doc_06, doc_01, doc_25",0,0.0,0.0,NaN
7,q08,Как работает css грид система?,doc_06,"doc_06, doc_23, doc_14",1,1.0,1.0,1.0
8,q09,Че такое cdn?,doc_26,"doc_26, doc_22, doc_12",1,1.0,1.0,1.0
9,q10,Чем является webAsembly,doc_24,"doc_24, doc_09, doc_13",1,1.0,1.0,1.0


,metric,value
0,mean_hit@3,0.90
1,mean_recall@3,0.90
2,mean_MRR@3,0.85


[{'query': 'Как работает сео аналитика для спа?', 'after_retrieved_sources': 'doc_06, doc_01, doc_25'}, {'query': 'Для чего нужен DOM манипуляции?', 'after_retrieved_sources': 'doc_17, doc_13, doc_23'}]
{'doc_id': 'doc_13', 'title': 'Сео(seo) аналитика для спа(spa) приложений', 'text': 'SEO (сео) веб-приложений (особенно SPA (спа) — Single Page Applications) работает через обеспечение индексации динамического контента, который создается JavaScript'} {'doc_id': 'doc_17', 'title': 'DOM манипуляции querySelector', 'text': 'DOM-манипуляции (Document Object Model) — это процесс изменения структуры, стиля или контента веб-страницы в реальном времени с помощью JavaScript'}


In [95]:
retrieval_before_after_update = []
retrieval_before_after_update.append({
    "query": retrieved_before[0]["query"],
    "before_retrieved_sources": retrieved_before[0]["before_retrieved_sources"],
    "after_retrieved_sources": retrieved_after[0]["after_retrieved_sources"],
    "changed": "Поменялись все, используемые документы",
})
retrieval_before_after_update.append({
    "query": retrieved_before[1]["query"],
    "before_retrieved_sources": retrieved_before[1]["before_retrieved_sources"],
    "after_retrieved_sources": retrieved_after[1]["after_retrieved_sources"],
    "changed": "Поменялись все, используемые документы",
})

retrieval_before_after_update_df = pd.DataFrame(retrieval_before_after_update, columns = ["query", "before_retrieved_sources", "after_retrieved_sources", "changed"])
retrieval_before_after_update_df.to_csv('artifacts/retrieval_before_after_update.csv', index=False, encoding='utf-8')

In [46]:
def build_context_from_retrieval(query: str, artifacts: RetrieverArtifacts, top_k: int = 3) -> Tuple[str, pd.DataFrame]:
    retrieved = search_chunks(query, artifacts=artifacts, top_k=top_k)
    context_blocks = []

    for _, row in retrieved.iterrows():
        block = (
            f"[Источник: {row['doc_id']} | {row['title']} | score={row['score']:.4f}]\n"
            f"{row['chunk_text']}"
        )
        context_blocks.append(block)

    context = "\n\n".join(context_blocks)
    return context, retrieved

In [47]:
query = "Зачем нужны promise?"
context, retrieved_df = build_context_from_retrieval(query, artifacts=artifacts_1, top_k=3)

display(Markdown(f"### Запрос: {query}"))
display(retrieved_df)
print(context)

### Запрос: Зачем нужны promise?

,rank,score,doc_id,title,chunk_id,chunk_text
0,1,0.370790,doc_08,JavaScript async/await,doc_08_chunk_0,async/await – синтаксический сахар над Promise...
1,2,0.194093,doc_01,Что такое HTTP протокол,doc_01_chunk_1,Работает по модели клиент-сервер: клиент (брау...
2,3,0.177825,doc_08,JavaScript async/await,doc_08_chunk_1,resolve. try/catch для обработки ошибок. Promi...


[Источник: doc_08 | JavaScript async/await | score=0.3708]
async/await – синтаксический сахар над Promise. async function всегда возвращает Promise. await приостанавливает выполнение до resolve. try/catch для обработки ошибок. Promise.all для параллельного выполнения. Деструктуризация:

[Источник: doc_01 | Что такое HTTP протокол | score=0.1941]
Работает по модели клиент-сервер: клиент (браузер) отправляет запрос (request), сервер отвечает ответом (response). Методы: GET (получение), POST (создание), PUT (обновление), DELETE (удаление). Статусы: 200 OK,

[Источник: doc_08 | JavaScript async/await | score=0.1778]
resolve. try/catch для обработки ошибок. Promise.all для параллельного выполнения. Деструктуризация: const { data } = await response.json().


In [48]:
def generate_answer_from_context(query: str, context: str, max_sentences: int = 2) -> str:
    raw_lines = [line.strip() for line in context.splitlines() if line.strip()]
    content_lines = [line for line in raw_lines if not line.startswith("[Источник:")]

    sentence_pool = []
    for line in content_lines:
        sentence_pool.extend(split_into_sentences(line))

    sentence_pool = [s for s in sentence_pool if len(s.split()) >= 4]

    if not sentence_pool:
        return "Недостаточно контекста для построения ответа."

    vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    matrix = vectorizer.fit_transform([query] + sentence_pool).toarray().astype(np.float32)

    query_vec = matrix[0]
    sentence_vecs = matrix[1:]

    query_norm = np.linalg.norm(query_vec) + 1e-12
    sent_norms = np.linalg.norm(sentence_vecs, axis=1) + 1e-12
    scores = (sentence_vecs @ query_vec) / (sent_norms * query_norm)

    ranked_idx = np.argsort(-scores)
    selected_sentences = []
    used_normalized = set()

    for idx in ranked_idx:
        sentence = sentence_pool[idx]
        normalized = sentence.lower().strip()
        if scores[idx] <= 0:
            continue
        if normalized in used_normalized:
            continue
        used_normalized.add(normalized)
        selected_sentences.append(sentence)
        if len(selected_sentences) >= max_sentences:
            break

    if not selected_sentences:
        return "В найденном контексте нет достаточно релевантного фрагмента для уверенного ответа."

    return " ".join(selected_sentences)


In [49]:
answer_example = generate_answer_from_context(query, context)
print(answer_example)

Promise.all для параллельного выполнения. async function всегда возвращает Promise.


In [50]:
def mini_rag_answer(
    query: str,
    artifacts: RetrieverArtifacts,
    top_k: int = 3,
    max_answer_sentences: int = 2,
) -> Dict[str, object]:
    context, retrieved = build_context_from_retrieval(query, artifacts=artifacts, top_k=top_k)
    answer = generate_answer_from_context(query, context=context, max_sentences=max_answer_sentences)

    return {
        "query": query,
        "answer": answer,
        "context": context,
        "sources": retrieved,
    }

In [51]:
rag_result = mini_rag_answer(
    "Зачем нужен promise?",
    artifacts=artifacts_1,
    top_k=3,
)

display(Markdown(f"### Вопрос: {rag_result['query']}"))
display(Markdown(f"**Ответ:** {rag_result['answer']}"))
display(Markdown("**Источники:**"))
display(rag_result["sources"])

### Вопрос: Зачем нужен promise?

**Ответ:** Promise.all для параллельного выполнения. async function всегда возвращает Promise.

**Источники:**

,rank,score,doc_id,title,chunk_id,chunk_text
0,1,0.364377,doc_08,JavaScript async/await,doc_08_chunk_0,async/await – синтаксический сахар над Promise...
1,2,0.184882,doc_01,Что такое HTTP протокол,doc_01_chunk_1,Работает по модели клиент-сервер: клиент (брау...
2,3,0.171961,doc_01,Что такое HTTP протокол,doc_01_chunk_0,HTTP (HyperText Transfer Protocol) – протокол ...


In [63]:
comparison_queries = [
    "Как работает cors?", 
    "Для чего нужна CEO аналитика страницы?", 
    "Как работает ассинхронность в JavaScript?", 
    "Что такое замыкания?"
]

comparison_rows = []

for query in comparison_queries:
    rag = mini_rag_answer(query, artifacts=artifacts_1, top_k=3)
    display(Markdown(f"### Вопрос: {query}"))
    
    comparison_rows.append({
        "question": query,
        "answer": rag.get('answer', 'Нет ответа'),
        "retrieved_sources": context,
    })
    
    display(Markdown(f"### Ответ: {rag.get('answer', 'Нет ответа')}"))
    print(context)
    
rag_examples_df = pd.DataFrame(comparison_rows, columns=["question", "answer", "retrieved_sources"])
rag_examples_df.to_csv('artifacts/rag_examples.csv', index=False, encoding='utf-8')
print(rag_examples_df)

### Вопрос: Как работает cors?

### Ответ: CORS (Cross-Origin Resource Sharing) – механизм безопасности браузера. Работает по модели клиент-сервер: клиент (браузер) отправляет запрос (request), сервер отвечает ответом (response).

[Источник: doc_08 | JavaScript async/await | score=0.3708]
async/await – синтаксический сахар над Promise. async function всегда возвращает Promise. await приостанавливает выполнение до resolve. try/catch для обработки ошибок. Promise.all для параллельного выполнения. Деструктуризация:

[Источник: doc_01 | Что такое HTTP протокол | score=0.1941]
Работает по модели клиент-сервер: клиент (браузер) отправляет запрос (request), сервер отвечает ответом (response). Методы: GET (получение), POST (создание), PUT (обновление), DELETE (удаление). Статусы: 200 OK,

[Источник: doc_08 | JavaScript async/await | score=0.1778]
resolve. try/catch для обработки ошибок. Promise.all для параллельного выполнения. Деструктуризация: const { data } = await response.json().


### Вопрос: Для чего нужна CEO аналитика страницы?

### Ответ: Coverage для неиспользуемого CSS/JS.

...

### Ответ: В найденном контексте нет достаточно релевантного фрагмента для уверенного ответа.

[Источник: doc_08 | JavaScript async/await | score=0.3708]
async/await – синтаксический сахар над Promise. async function всегда возвращает Promise. await приостанавливает выполнение до resolve. try/catch для обработки ошибок. Promise.all для параллельного выполнения. Деструктуризация:

[Источник: doc_01 | Что такое HTTP протокол | score=0.1941]
Работает по модели клиент-сервер: клиент (браузер) отправляет запрос (request), сервер отвечает ответом (response). Методы: GET (получение), POST (создание), PUT (обновление), DELETE (удаление). Статусы: 200 OK,

[Источник: doc_08 | JavaScript async/await | score=0.1778]
resolve. try/catch для обработки ошибок. Promise.all для параллельного выполнения. Деструктуризация: const { data } = await response.json().


### Вопрос: Что такое замыкания?

### Ответ: В найденном контексте нет достаточно релевантного фрагмента для уверенного ответа.

[Источник: doc_08 | JavaScript async/await | score=0.3708]
async/await – синтаксический сахар над Promise. async function всегда возвращает Promise. await приостанавливает выполнение до resolve. try/catch для обработки ошибок. Promise.all для параллельного выполнения. Деструктуризация:

[Источник: doc_01 | Что такое HTTP протокол | score=0.1941]
Работает по модели клиент-сервер: клиент (браузер) отправляет запрос (request), сервер отвечает ответом (response). Методы: GET (получение), POST (создание), PUT (обновление), DELETE (удаление). Статусы: 200 OK,

[Источник: doc_08 | JavaScript async/await | score=0.1778]
resolve. try/catch для обработки ошибок. Promise.all для параллельного выполнения. Деструктуризация: const { data } = await response.json().
                                    question  \
0                         Как работает cors?   
1     Для чего нужна CEO аналитика страницы?   
2  Как работает ассинхронность в JavaScript?   
3                       Что такое замыкания?   

## Что пошло не так? 

Все ответы кроме первого не относятся к теме, в вопрос про ассинхроность javascript и замыкания, смею предположить, что не нашло контекста в предоставленных документах, т.к. слова не совпали с тем, что написано в тексте, хоть они и являются ими

В вопросе про сео аналитику он ответил абсолютно по другой теме 

Мое предположение что проблема кроется в реализации retrieval и составе контекста 